In [89]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "raw"

sys.path.insert(0, str(PROJECT_DIR / "src"))

print("Project folder:", PROJECT_DIR)

Project folder: /Users/ramun/Desktop/Uni/HS26/Data Analysis/project/f1-project


# Analysis

In [90]:
import pandas as pd


race_results = pd.read_csv(
    DATA_DIR / "f1db-races-race-results.csv"
)
races = pd.read_csv(DATA_DIR / "f1db-races.csv")
quali_results = pd.read_csv(
    DATA_DIR / "f1db-races-qualifying-results.csv"
)
circuits = pd.read_csv(DATA_DIR / "f1db-circuits.csv")


races = races[races["year"].between(2010, 2026)]
race_results = race_results[race_results["year"].between(2010, 2026)]
quali_results = quali_results[quali_results["year"].between(2010, 2026)]

# One row per race
# Subset focuses only on the id -> race. Sum then adds up all the True values from duplicated() which only occur if an id is there for more than one time
print("Duplicate races:", races.duplicated(subset=["id"]).sum())

# One row per driver and race
print("Duplicate race results:", race_results.duplicated(subset=["raceId", "driverId"]).sum())

print("Duplicate qualifying results:", quali_results.duplicated(subset=["raceId", "driverId"]).sum())



Duplicate races: 0
Duplicate race results: 0
Duplicate qualifying results: 0


/var/folders/y6/yvr9h5p110qc_zbybx4w5p680000gn/T/ipykernel_27660/1327182250.py:4: DtypeWarning: Columns (31: fastestLap, 32: driverOfTheDay) have mixed types. Specify dtype option on import or set low_memory=False.
  race_results = pd.read_csv(


- Races: Checks whether a race ID appears more than once.
- Race results: Checks whether a driver appears more than once in the same race.
- Qualifying results: Checks whether a driver appears more than once in the same qualifying session.

Especially important for new data updates

## Merge data sets

In [91]:
# how="inner" means that only for available ids of race_results, the values of race get joined (not too important here since the data is already very cleanly prepared)
# validate: raceId could be contained multiple times but Id must only appear once, else a MergeError would be triggered
# drop: drops id since we have raceId and id which are the same

model_data = race_results.merge(
    races[[
        "id",
        "date",
        "circuitId",
        "circuitType",
        "courseLength",
        "turns",
        "scheduledLaps"]],
    left_on="raceId",
    right_on="id",
    how="inner",
    validate="many_to_one"
).drop(columns="id")


# on: assigns to every driver the qualifying variables to the according race
# left: all the data of model_data stays, where the suiting parts from quali results get added
# rename is needed, since positionNumber is in the model_data used for the final position

model_data = model_data.merge(
    quali_results[[
        "raceId",
        "driverId",
        "q1Millis",
        "q2Millis",
        "q3Millis",
    ]],
    on=["raceId", "driverId"],
    how="left",
    validate="one_to_one"
)



## How to deal with missing values in gridPositionNumber

In [92]:
print(model_data["gridPositionText"].unique())

gridPositionNumber_nan = model_data[model_data["gridPositionText"].isna()]

<ArrowStringArray>
[ '3',  '2',  '4',  '1',  '5',  '7',  '8',  '6', '12', '11',  '9', '10', '18',
 '13', '21', '15', '20', '14', 'PL', '19', '17', '16', '22', '24', '23',  nan]
Length: 26, dtype: str


All the nan values are because of DNS or DNQ, thus they should be excluded. PL stands for "pit lane", meaning that the drivers had to start from the pit lane, which is more or less euqivalent as starting from the last position. Since the field hasn't always the same size, it can't be just set to 20.

In [93]:
model_data = model_data[model_data["gridPositionText"].notna()]

# transform("size"): size counts rows per group, where raceId is treated as the group

model_data["fieldSize"] = model_data.groupby("raceId")["driverId"].transform("size")


# .loc searches for the column gridPositionText with value "PL" and takes the related gridPositionNumber

model_data.loc[ model_data["gridPositionText"] == "PL", "gridPositionNumber"] = model_data["fieldSize"]

## Exlude and add features

The last command selects only those columns wanted 

In [94]:
columns_needed = [
    "raceId",
    "fieldSize",
    "year",
    "round",
    "date",
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType",
    "courseLength",
    "turns",
    "scheduledLaps",
    "gridPositionNumber",
    "gridPositionText",
    "qualificationPositionNumber",
    "q1Millis",
    "q2Millis",
    "q3Millis",
    "positionDisplayOrder",
    "positionText",
    "reasonRetired"
]


model_data = model_data[columns_needed]

Normalizing positions makes races with different field sizes comparable: finishing 20th out of 20 and 20th out of 22.

In [95]:
model_data["qualifying_fraction"] = (
    (model_data["qualificationPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["grid_fraction"] = (
    (model_data["gridPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["finish_fraction"] = (
    (model_data["positionDisplayOrder"] - 1)
    / (model_data["fieldSize"] - 1)
)

#### 5 race average

In [96]:
# transform: applies a computation on every group and returns its results back to the original row.
# shift(1): shifts all values one rows downwards, such that the result only gets included in the next races last five average
# rolling(5): uses five consecutive values
# min_periods says that the value should also be computed if only 4 instead of 5 values are available

model_data["driver_finish_5"] = (
    model_data.groupby("driverId")["finish_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)


#### 5 quali average

In [97]:
model_data["driver_quali_5"] = (
    model_data.groupby("driverId")["qualifying_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

#### Quali deviation feature

In [98]:
model_data["quali_vs_recent"] = (
    model_data["qualifying_fraction"]
    - model_data["driver_quali_5"]
)

#### Failure rate

In [99]:
model_data["failure"] = (model_data["positionText"].eq("DNF") | model_data["reasonRetired"].notna())

# .eq("DNF") is the same as == "DNF"

# .notna() returns True when a retirement reason is recorded.
# A missing reason alone does not rule out DNF; the first condition checks that.
# .notna() gives back True, if a value isn't missing. (Everything that is missing in reasonRetiered hasn't retiered -> should return False)

model_data["failure_rate"] = (model_data.groupby("driverId")["failure"]
                                .transform(lambda failrate: failrate.shift(1).rolling(10, min_periods=5).mean()))

# mean values True and False as 0 and 1 leading to e.g. 0.4 -> 40%

#### Adding other average historical features

In [100]:
for window in [3, 5, 10, 20]:
    for source, name in [("finish_fraction", "finish"), ("qualifying_fraction", "quali"), ("failure", "failure_rate")]:
        model_data[f"driver_{name}_{window}"] = (model_data.groupby("driverId")[source]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))

This adds the average values not of the features computed above also for other race windows. i.e. 3, 5, 10 and 20 races

#### Standard deviations: consistency in races

In [101]:
for window in [5, 10]:
    model_data[f"driver_std_{window}"] = (
        model_data.groupby("driverId")["finish_fraction"].transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
    )

#### Several features for driver experience

Total experience in races

In [102]:
model_data["driver_experience"] = model_data.groupby("driverId").cumcount()       # cumcount() counts the total apperances up to this point


Experience on this exact circuit (Possibly important if a new one gets intorduced)

In [103]:
model_data["driver_circuit_experience"] = model_data.groupby(["driverId", "circuitId"]).cumcount()


How good a driver performs on a circuit. Once again only taking the previous 3 results into account, with a minimum of 1 race. If never raced this circuit before the drivers get a NaN, which later must be changed. Maybe to the median of the whold field or just ignored.

In [104]:
model_data["driver_circuit_form"] = (
    model_data.groupby(["driverId", "circuitId"])["finish_fraction"]
    .transform(lambda x:
                x.shift(1).rolling(3, min_periods=1).mean())
)

How often does the driver make a mistake/dnf?

In [105]:
for window in [3, 5, 10, 20]:
    model_data[f"driver_dnf_{window}"] = (
        model_data.groupby("driverId")["failure"]
        .transform(lambda x:
                    x.shift(1).rolling(window, min_periods=1).mean()))

#### Team Performance

We create a separate team table with one row per team per race, so rolling averages count races rather than individual drivers. Later these data sets get merged together

In [106]:
team_data = (
    model_data.groupby(
        ["raceId", "constructorId", "circuitId"],
        as_index=False                          # to ensure that the correct indizes remain, which is important for later merges
    )
    .agg(                                       # compresses each constructor per race into a row and creates the columns
        team_finish=("finish_fraction", "mean"),        # new column name = used variables, method
        team_quali=("qualifying_fraction", "mean"),
        team_dnf=("failure", "mean")
    )
)

Now done for different windows in history. Once again only using previous values (shift(1))

In [107]:
for window in [3, 5, 10, 20]:
    for source in ["team_finish", "team_quali", "team_dnf"]:
        team_data[f"{source}_{window}"] = (
            team_data.groupby("constructorId")[source]
            .transform(lambda x: 
                       x.shift(1).rolling(window, min_periods=1).mean()))

Now the average team form on a certain circuit is added to the team_data table

In [108]:
team_data["team_circuit_form"] = (
    team_data.groupby(["constructorId", "circuitId"])["team_finish"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

Finally we merge these features to the original data set.

In [109]:
# create a list via list comprehension with all the names of the team features
team_features = [f"{name}_{window}" for window in [3, 5, 10, 20] for name in ["team_finish", "team_quali", "team_dnf"]] + ["team_circuit_form"]


model_data = model_data.merge(
    team_data[["raceId", "constructorId"] + team_features],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one"
)

#### Differences

Between Driver and Teammate

In [110]:
model_data["driver_team_difference"] = (
    model_data["driver_finish_5"] - model_data["team_finish_5"]
)

Team Qualifying vs recent Team performance

In [111]:
model_data["team_quali_vs_recent"] = (
    model_data.groupby(["raceId", "constructorId"])["qualifying_fraction"].transform("mean") - model_data["team_quali_5"]
)

#### Qualifying Features

Gap to the leader in the session in %. If already droped out, a NaN will be created!

In [112]:
for stage in [1, 2, 3]:
    column = f"q{stage}Millis"

    best_time = model_data.groupby("raceId")[column].transform("min")       # min is here the minimum, not minute as one might think because of time feature

    model_data[f"q{stage}_gap"] = (model_data[column] / best_time - 1) * 100

#### Create a factor variable for street tracks

In [113]:
model_data["street"] = (
    model_data["circuitType"].eq("STREET").astype(int)          # checks if the street type is equal to STREET. If so it assigns the variable 1
)

Create different starting and qualifying variables for street tracks. This could be particularly important because street tracks are known for their narrowness, which makes it even more difficult to overtake. With those additional variables the model can account for that. If it isn't a street circuit the variable is equal to 0 which automatically eliminates it out of the model. Thus, it works like a weighted factor variable.

In [114]:
model_data["grid_street"] = (
    model_data["grid_fraction"] * model_data["street"]
)

model_data["qualifying_street"] = (
    model_data["qualifying_fraction"] * model_data["street"]
)

#### Pit-Lane Start
The model should be able to give a start from the pit lane a different weight.

In [115]:
model_data["pit_lane"] = model_data["gridPositionText"].eq("PL").astype(int)

#### Failure Rates on circuit
Again we create a new data set to use rolling for the last 5 races, not driver performances

In [116]:
circuit_data = (
    model_data.groupby(["raceId", "circuitId"], as_index=False)
    .agg(circuit_dnf=("failure", "mean"))                           # create the variable circuit_dnf which is the mean failure rate from the failure feature
)


# use the last 5 races on this circuit to create the rolling average
circuit_data["circuit_dnf_history"] = (
    circuit_data.groupby("circuitId")["circuit_dnf"]
    .transform(lambda x:
                x.shift(1).rolling(5, min_periods=1).mean())
)


# merge the data into the original data set
model_data = model_data.merge(
    circuit_data[["raceId", "circuit_dnf_history"]],
    on="raceId",
    how="left",
    validate="many_to_one"
)

#### Compare the features here with the ones from the skript

No need for anymore, as we now produce different features by intent

In [117]:
# import importlib
# from models import features

# # Reload the module so saved changes to features.py are used.
# importlib.reload(features)

# # calls the prepare_features function from the features.py file
# prepared_data = features.prepare_features()

# notebook_data = model_data.copy()

# script_data = prepared_data

# # tests, if the data sets are equivalent
# pd.testing.assert_frame_equal(notebook_data, script_data)

# print("Script and notebook produce the same features")

## Model feature selection

#### Categorical features

In [118]:
categorical_features = [
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType"
]

#### Numerical features

In [119]:
numeric_features = [
    "gridPositionNumber",
    "qualificationPositionNumber",
    "grid_fraction",
    "qualifying_fraction",
    "pit_lane",
    "street",
    "grid_street",
    "qualifying_street",
    "courseLength",
    "turns",
    "scheduledLaps",
    "q1_gap",
    # "q2_gap",
    # "q3_gap",
    "driver_std_5",
    "driver_std_10",
    "driver_experience",
    "driver_circuit_form",
    "driver_circuit_experience",
    "team_circuit_form",
    "circuit_dnf_history",
    "driver_team_difference",
    "quali_vs_recent",
    "team_quali_vs_recent"
]

rolling_window_features = ["driver_finish","driver_quali","driver_dnf","team_finish","team_quali","team_dnf"]

for window in [3, 5, 10, 20]:
    for name in rolling_window_features:
        numeric_features.append(f"{name}_{window}")

q2_gap and q3_gap are left out because their missing values often indicate elimination in an earlier qualifying stage, making median imputation potentially misleading.

## Split Data for CV and Final Test

In [120]:
# This data is used to find the best model via CV
development_data = model_data[model_data["year"].between(2010, 2024)].copy()


# This data is for the final test
test_data = model_data[model_data["year"] == 2025].copy()

#### CV rolling window for 5 years

myb 10 years could be better...?

In [121]:
cv_splits = []

for validation_year in [2022, 2023, 2024]:
    train = development_data[development_data["year"].between(validation_year - 10, validation_year - 1)].copy()

    valid = development_data[development_data["year"] == validation_year].copy()

    cv_splits.append({
        "validation_year": validation_year,
        "train": train,
        "valid": valid})

## Early Model Brainstorm
1. Local polynomial regression: Could flexibly model a few relationships, such as starting versus finishing position, but becomes less practical with many features.

2. Random Forest: Could capture nonlinear patterns and interactions without specifying them manually.
3. Gradient boosting / XGBoost: Could combine many small decision trees to capture subtle patterns in structured race data.
4. Ridge regression: Could suit our many correlated features, such as qualifying position and recent form. Regularization may reduce overfitting.

5. Mixed-effects models: Could account for repeated observations of the same drivers, teams, and circuits through group-specific effects.

## Model Processing 

We use a preprocessing pipeline to fill missing values, standardize numerical features, and encode categorical features.


Fill missing values: Some information is unavailable for example when a driver has never raced at a circuit before. Since some machine learing models cannot process missing values, we replace missing numbers with the median of that feature in the training data. We also add an indicator so the model can distinguish missing information from an actual recorded value.


Standardize numerical features: Features use different scales, such as circuit length in kilometres and failure rates between 0 and 1. We rescale them using their training mean and standard deviation so machine learnign models penaltizes  features comparably, regardless of their units. Especially if we use a ridge model this would be particularilly important, else this could be ignored.


Encode categorical features: MLs needs numbers rather than labels such as driver or team names. We create separate 0/1 columns for each category, indicating which driver, team, or circuit a row belongs to without implying an artificial numerical ranking.

For example, instead of assigning Ferrari = 1, McLaren = 2, and Mercedes = 3—which could create 


Team	    Ferrari	McLaren	Mercedes


Ferrari	    1	    0	    0


McLaren	    0	    1	    0


Mercedes	0	    0	    1


This identifies the team without implying that one team is numerically “greater” than another.


These steps are fitted only on the training data in each cross-validation fold and then applied to the validation data, preventing data leakage.

In [122]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# SimpleImputer is a class that estimates NaN values. Here they get replaced with the median value
# StandardScaler creates an object to standardize as explained above. It processes the data after the imputer: subtracts the mean and devides with the std. deviation
# Pipeline is used to execute step after step. Each step is a tuple with its name and what its method
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                             ("scaler", StandardScaler())])


# OneHotEncoder creates the described factor variables to distinguish things like constructors with numbers not with names
# handle_unknown decides how to deal with e.g. drivers who never were in the training data. All the created collumns are thus set to 0 for this e.g. driver
categorical_pipeline = Pipeline([ #("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),       probably not necessary because our data is pretty clean
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])


# The preprocessor defines how each feature will be prepared once data is passed in: numerical features are filled and scaled, while categorical features are converted into 0/1 columns.
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])

## 

## Model Training and Validation 

#### Linear Regression

In [123]:
from sklearn.linear_model import LinearRegression
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error


# Creates the pipeline for preprocessing the date in the first step with the model processig step. 
# Afterwards in the step regression a linear regression is fitted
model = Pipeline([("preprocessor", preprocessor), ("regression", LinearRegression())])



# create an empty list to later store the MAE of each iteration
cv_results = []


# Take the cv_splits list from the section above and create for every split a train and validation split

# split 1: 2017 - 21 train, 2022 valid
# split 2: 2018 - 22 train, 2023 valid
# split 3: 2019 - 23 train, 2024 valid
for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])


    # each fold trains its own model. Clone clones the whole pipeline from model, s.t. no model learns from the others
    fold_model = clone(model)

    # here we fit the model. this is done by giving two parameters. firstly all the numeric and categorical features
    # then declaring which variable is the depending one.
    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])


    # here the prediction is made. this is done by only providing the features of the validation set. 
    # with our previous setted settings the predicted values should be the finish_fraction. 
    # Obviously the smaller the value, the better the predicted placement.
    predictions = fold_model.predict(valid[numeric_features + categorical_features])


    # After the last step the predictions are some arbitrary values without a concrete predicited place. 
    # Thus, now these "score values" get ranked. As stated above, the smaller the value, the better the predicted placement.
    # Therefore, we will rank in an acsending manner: smalles score -> place no. 1, etc.

    # pd.Series is exactly used for creating such series. predictions are the values that need to be ranked by the series.
    # index = valid.index ensures that each predction is assigned to its original driver
    # groupby(valid["raceId"]) makes sure that each raced is analyzed on its own
    # rank(method="first") then finally assigns the places within a race form 1 ascending
    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")


    # this constructs our statistic to compare the quality of our model.
    #  The mean absolute error is computed by ∑|true placement - predicted placement|/(dirvers * races)
    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)


    # append the result to the list, for each year creating a dict
    cv_results.append({"validation_year": split["validation_year"],"mae": mae})


# creates a pandas data set for the cv_results
cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,3.671233
1,2023,3.409611
2,2024,3.060543


Average validation MAE: 3.380


In [124]:
# store the result so that it won't get overwritten by the next model
linear_results = cv_results_df.copy()

#### Random Forest

In [125]:
from sklearn.ensemble import RandomForestRegressor

# a new model pipeline is declared: preprocessing still is the same process of replacing NaNs etc.
# but now the regression is done by a random forest.
# n_estimators: Trains n decision trees and averages over their predictions
# min_sample_leaf: Each finalized group of a tree must contain at leat n observations. Here mae is optimized with n = 31
# ranodm_state: same as set seed, used to make results reproducable
# n_jobs: used to declare which cpu cores get used (idk)
model = Pipeline([("preprocessor", preprocessor),
    ("regression", RandomForestRegressor(n_estimators=300, min_samples_leaf=31, random_state=13, n_jobs=-1))])

# same as in the linear regression
cv_results = []

for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])

    fold_model = clone(model)

    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])

    predictions = fold_model.predict(valid[numeric_features + categorical_features])

    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")

    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)

    cv_results.append({"validation_year": split["validation_year"],"mae": mae})

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,3.557078
1,2023,3.272311
2,2024,2.960334


Average validation MAE: 3.263


In [126]:
forest_results = cv_results_df.copy()

#### xgboost

The trees get trained one after the other, trying to correct the mistakes the trees before have done.

In [127]:
from xgboost import XGBRegressor

# n_estimators: n trees
# learing_rate: each tree only corrects a small part to make the learning process more carefull
# max_depth: each tree can at most do n splits
# after each split, each group must at least contain n observations
# subsample: each tree can only use this fraction of the training set. the partition is random
# colsample_bytree: each tree can only use this fraction of features
# reg_lambda: Higher values penalize large leaf weights more strongly, especially in small leaves.
# objective: declares how to measure errors and how to minimize them
# tree_method: optimizes speed by apporximation via sepperation in bins. "auto" is the default value
model = Pipeline([("preprocessor", preprocessor),
    ("regression", XGBRegressor(n_estimators=300, learning_rate=0.02, max_depth=20, 
                                min_child_weight=100, subsample=0.6, colsample_bytree=0.6, 
                                reg_lambda=9, objective="reg:absoluteerror", tree_method="auto", 
                                random_state=0, n_jobs=2))])

# try different approaches for the objective e.g. absoluteerror and squarederror
# at first glance abs error seems to perform better

cv_results = []

for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])

    fold_model = clone(model)

    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])

    predictions = fold_model.predict(valid[numeric_features + categorical_features])

    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")

    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)

    cv_results.append({"validation_year": split["validation_year"],"mae": mae})

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,3.543379
1,2023,3.359268
2,2024,2.926931


Average validation MAE: 3.277


In [128]:
xgboost_results = cv_results_df.copy()

##### Searching the best configutarion of parameter settings

Optuna uses Bayesian optimization to search for suitable model parameters. It uses previous trial results to suggest promising configurations, trying to minimize the average CV mae across multiple years and random seeds.

In [136]:
import optuna
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# both feature groups get unified
# each parameter configuration is tested on 3 seeds: 0, 13, 42
features = numeric_features + categorical_features
seeds = [0, 13, 67]


# Optuna uses this function to evaluate each parameter configuration
# a trial represents one optimization attempt and suggests the parameters to test
def objective(trial):

    # Defines the range of values Optuna can search through for each parameter
    parameters = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1200, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 5),
        "min_child_weight": trial.suggest_float("min_child_weight", 5, 200, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0), 
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0), 
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 100, log=True)
        }

    # list to store the 3 maes per iteration
    seed_maes = []

    # evaluates for each seed
    for seed in seeds:

        # yearly is not really an accurate name but I had no better idea: It stores the mae of each spilt
        yearly_maes = []
        # takes the defined data splits
        for split in cv_splits:
            train = split["train"]

            # this is an incredible important addition, to make sure, that we don't use information of the races based on the order of the drivers in the data set. 
            # This makes sure that the drivers are sorted in a non-biased way, e.g. ther driverId/name
            valid = split["valid"].sort_values(["raceId", "driverId"])

            # create a new Pipeline for each candidate, which represents some "arbitrary" setting of the parametes
            # here we need to clone the preprocessor, which happens above in the clone(model), 
            # to ensure he computes every time the new medians, etc. for each split
            # **parameters: ** distributes the parameters to all requiered spots. sort of unpacks it in the function call
            # the parametes that weren't already declared in the "parameters"
            candidate = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("regression", XGBRegressor(**parameters, objective="reg:absoluteerror", 
                                            tree_method="auto", random_state=seed, n_jobs=2))])

            # just as in the upper models values are fitted/ model gets trained with the train sets
            candidate.fit(train[features], train["finish_fraction"])

            # and predictions are made
            predictions = pd.Series(candidate.predict(valid[features]), index=valid.index)

            predicted_positions = predictions.groupby(valid["raceId"]).rank(method="first")

            # for each split the mae is appended
            yearly_maes.append(mean_absolute_error(valid["positionDisplayOrder"], predicted_positions))

        # the average for each seed is created and appended to the list
        seed_maes.append(sum(yearly_maes) / len(yearly_maes))

    # store additional information in each trial of optuna
    # firstly the seed maes and additionaly the standard deviation

    # this can be heloful, if we later want to call for e.g. the best trials attributes
    trial.set_user_attr("seed_maes", seed_maes)
    trial.set_user_attr("seed_std", float(pd.Series(seed_maes).std()))


    # returns the value optuna should optimize: the average over all seeds, which
    return sum(seed_maes) / len(seed_maes)

# direction = minimize declares, that the return value abouve should get minimized -> find the smallest MAE possible
# sampler=TPESamper: seed: "controlls" the ranomization,
# n_startup_trials: defines how many completly random settings are made to explore the data set
# afterwards the sampler parts the findings in a group with good mae values
# later it estimates which parameters are how distributed in which groups ~~~
# at last it recommends futher vaues which should be more likely
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=13, n_startup_trials=30))

# Uses the existing configuration as the starting reference
study.enqueue_trial({'n_estimators': 100, 'learning_rate': 0.005318421512175779, 
                     'max_depth': 1, 'min_child_weight': 6.607289473852337, 
                     'subsample': 0.9850072285777673, 'colsample_bytree': 0.7925494017883324, 
                     'reg_lambda': 0.11796923067693268})


# calls our defined optimize function and repeats it 150 times
study.optimize(objective, n_trials=150, n_jobs=1)

print("Best CV MAE:", round(study.best_value, 3))
print("Best parameters:", study.best_params)
print("MAE per seed:", study.best_trial.user_attrs["seed_maes"])
print("Seed standard deviation:", study.best_trial.user_attrs["seed_std"])

# shows the best 10 configurations
display(study.trials_dataframe().sort_values("value").head(10))

[I 2026-11-19 16:24:24,636] A new study created in memory with name: no-name-fbcae2c5-f9d0-4f13-8044-ba428644d734
[I 2026-11-19 16:24:25,330] Trial 0 finished with value: 3.604852589537201 and parameters: {'n_estimators': 100, 'learning_rate': 0.005318421512175779, 'max_depth': 1, 'min_child_weight': 6.607289473852337, 'subsample': 0.9850072285777673, 'colsample_bytree': 0.7925494017883324, 'reg_lambda': 0.11796923067693268}. Best is trial 0 with value: 3.604852589537201.
[I 2026-11-19 16:24:33,948] Trial 1 finished with value: 3.287098654224206 and parameters: {'n_estimators': 950, 'learning_rate': 0.011216237307962802, 'max_depth': 5, 'min_child_weight': 176.26178186548532, 'subsample': 0.9863005569524467, 'colsample_bytree': 0.7267246237086561, 'reg_lambda': 6.716258269109578}. Best is trial 1 with value: 3.287098654224206.
[I 2026-11-19 16:24:42,519] Trial 2 finished with value: 3.313429809365402 and parameters: {'n_estimators': 950, 'learning_rate': 0.04433132512822792, 'max_depth

KeyboardInterrupt: 

The baseline predicts that drivers will finish in their starting order. It provides a simple benchmark to check whether our machine learning models improve predictions beyond the information already contained in the starting grid.

In [130]:
baseline_results = []

for split in cv_splits:
    valid = split["valid"].sort_values(["raceId", "driverId"])

    # here we profit from our earlier declaration of how to trat missing values in grid position.
    # now we don't need to do any kind of filtering etc.
    scores = valid["gridPositionNumber"]
    
    predicted_positions = scores.groupby(valid["raceId"]).rank(method="first")

    baseline_results.append({
        "validation_year": split["validation_year"], 
        "mae": mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)})

baseline_results_df = pd.DataFrame(baseline_results)

display(baseline_results_df)

print("Average baseline MAE:",round(baseline_results_df["mae"].mean(), 3))

,validation_year,mae
0,2022,3.954338
1,2023,3.729977
2,2024,2.893528


Average baseline MAE: 3.526


## 2025 Evaluation

In [131]:
train_final = model_data[model_data["year"].between(2014, 2024)].copy()

# splited in the split for CV section
test_final = test_data.sort_values(["raceId", "driverId"])

features = numeric_features + categorical_features

# takes the best parameters from the optuna study
best_parameters = study.best_params.copy()


test_results = []
# prepares a list to store the later predictions
test_predictions = test_final[["raceId", "driverId", "positionDisplayOrder"]].copy()

for seed in [0, 13, 123]:

    # again prepare for each seed a new clean pipeline
    final_model = Pipeline([
        ("preprocessor", clone(preprocessor)),

        # unpack the best parameters into the regresison function and a add the missing parameters
        ("regression", XGBRegressor(**best_parameters, objective="reg:absoluteerror", 
                                    tree_method="auto", random_state=seed, n_jobs=2))])

    # fit the model
    final_model.fit(train_final[features], train_final["finish_fraction"])


    # predict the positions scores
    predictions = pd.Series(final_model.predict(test_final[features]), index=test_final.index)

    # rank the scores ascending
    positions = predictions.groupby(test_final["raceId"]).rank(method="first")

    # store the predictions in the according seed column for potential further investigation
    test_predictions[f"prediction_seed_{seed}"] = positions

    # store the mean average error in the result list
    test_results.append({"seed": seed, "mae": mean_absolute_error(test_final["positionDisplayOrder"], positions)})

# create the baseline model on the same 2025 observations
baseline_scores = (test_final["gridPositionNumber"])

baseline_positions = baseline_scores.groupby(test_final["raceId"]).rank(method="first")

baseline_mae = mean_absolute_error(test_final["positionDisplayOrder"], baseline_positions)

# create an other baseline (which some studies argue is the better one) to check our model with

quali_scores = test_final["qualificationPositionNumber"].fillna(float("inf"))       # float("inf") makes sure that drivers with missing values get appended at the end of the grid
quali_positions = quali_scores.groupby(test_final["raceId"]).rank(method="first")   # creates the prediction

quali_mae = mean_absolute_error(test_final["positionDisplayOrder"], quali_positions)


test_results_df = pd.DataFrame(test_results)
display(test_results_df)

print("Average XGBoost MAE:", round(test_results_df["mae"].mean(), 3))
print("Seed standard deviation:", round(test_results_df["mae"].std(), 3))
print("Startposition baseline MAE:", round(baseline_mae, 3))
print("Qualifying baseline MAE:", round(quali_mae, 3))

NameError: name 'study' is not defined

## Walk-forward Evaluation

Now as we roughly know which model type seems to be promising, we try to predict the 2025 races with ongoing generation of new models, to simulate a "real" application of our future model. This is called walk-forward evaluation, since for every race weekend we estimate a new model. For efficiency reasons for now we leave the hyperparameter as they are.

In [132]:
walk_data = model_data.copy()           # .copy() to leave the original data as it is
walk_data["date"] = pd.to_datetime(walk_data["date"])   # pd.to_datetime converts the dates in the column to real date value. This is especially important to set date bound

features = numeric_features + categorical_features


def walk_forward(model, year):
    walk_results = []

    # .eq marks the wanted rows, .loc then selects them
    evaluation_data = walk_data.loc[walk_data["year"].eq(year)]

    # Iterating over groupby returns a tuple for each race:
    # (race ID, data frame with all driver rows and columns for that race)
    for race_id, race in evaluation_data.groupby("raceId"):

        race_date = race["date"].iloc[0]                   # for every race it takes the date form the first row, since all the dates are the same, since they stem form the same raceId
        window_start = race_date - pd.DateOffset(years=10)  # for each race, the training data window is set to all races from 5 years ago

        # .ge(): on or after the given date
        # .lt(): strictly before the given date
        # The training window starts five years before the race and excludes the race being predicted.
        train = walk_data.loc[walk_data["date"].ge(window_start) & walk_data["date"].lt(race_date)]

        # Use a tie-break that does not depend on the actual finishing order!
        # This is important since the drivers are sorted in the data acoording to their actual finishing position.
        # To prevent that we take any information, we must sort the drivers within that race in an unbiased order, e.g. their driverId / names
        race = race.sort_values("driverId")

        # Clone the entire pipeline to start with an unfitted model for each race.
        race_model = clone(model)

        # finish_fraction ~ features
        race_model.fit(train[features], train["finish_fraction"])

        scores = pd.Series(race_model.predict(race[features]), index=race.index)    # assigns each driver its predicted score
        positions = scores.rank(method="first")         # ranks the drivers according to their score

        # Creates a data frame with predictions and errors for this race
        walk_results.append(pd.DataFrame({
            "raceId": race_id, "driverId": race["driverId"], "positionDisplayOrder": race["positionDisplayOrder"],
            "predicted_position": positions, "absolute_error": (race["positionDisplayOrder"] - positions).abs()}))

    # UPDATED: Combines the individual race tables for this model and year.
    return pd.concat(walk_results, ignore_index=True)

#### Baseline Models

In [133]:
baseline_results = []

for year in [2022, 2023, 2024]:

    # Creates the data structure for the baseline check for each year.
    evaluation_data = walk_data.loc[walk_data["year"].eq(year)].sort_values(["raceId", "driverId"])

    # Starting-grid baseline
    # .float("inf"): appends missing values at the end (tie breaks solved through sort with driverId!)
    grid_scores = evaluation_data["gridPositionNumber"].fillna(float("inf"))
    grid_positions = grid_scores.groupby(evaluation_data["raceId"]).rank(method="first")

    baseline_results.append({"model": "Starting-grid baseline",
                             "year": year,
                             "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], grid_positions)})

    # Qualifying baseline
    quali_scores = evaluation_data["qualificationPositionNumber"].fillna(float("inf"))
    quali_positions = quali_scores.groupby(evaluation_data["raceId"]).rank(method="first")

    baseline_results.append({"model": "Qualifying baseline",
                             "year": year,
                             "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], quali_positions)})

# pivot: changes here long data into wide. 
# index: which values become rows
# columns: which values become columns
# values: which values are in the cells 
baseline_comparison = pd.DataFrame(baseline_results).pivot(index="model", columns="year", values="mae")

baseline_comparison["Average MAE"] = baseline_comparison.mean(axis=1)

display(baseline_comparison.round(3))

year,2022,2023,2024,Average MAE
model,,,,
Qualifying baseline,3.653,3.657,2.823,3.377
Starting-grid baseline,3.954,3.730,2.894,3.526


#### xgboost & ridge

In [134]:
from sklearn.linear_model import Ridge

# create a new pipeline which uses once again the cloned, i.e. clean preprocessor and now as regression model the ridge model
# alpha: hyperparameter to punish large coefficients, the larger the harder the punishment
ridge_pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=300))])

validation_results = []

for year in [2022, 2023, 2024]:
    # use the walk_forward function from above to predict the results with ridge_pipeline as model
    ridge_predictions = walk_forward(ridge_pipeline, year=year)

    # ["absolute_error"].mean(): "absolute_error" is created automatically in our walk_forward function
    validation_results.append({"model": "Ridge", "year": year, "mae": ridge_predictions["absolute_error"].mean()})

    xgboost_seed_maes = []

    # for every seed we fit and predict the results with the hyperparameter from the optuna study
    for seed in [0, 13, 123]:
        xgboost_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("regression", XGBRegressor(**best_parameters, objective="reg:absoluteerror",
                tree_method="auto", random_state=seed, n_jobs=2))])

        # also for this model we use our own function to predict the results but with the xgboost_pipeline
        xgboost_predictions = walk_forward(xgboost_pipeline, year=year)

        xgboost_seed_maes.append(xgboost_predictions["absolute_error"].mean())

    validation_results.append({"model": "XGBoost", "year": year, "mae": sum(xgboost_seed_maes) / len(xgboost_seed_maes)})

comparison = pd.DataFrame(validation_results).pivot(index="model", columns="year", values="mae")

comparison["Average MAE"] = comparison.mean(axis=1)

display(comparison.sort_values("Average MAE").round(3))

NameError: name 'best_parameters' is not defined

Optimize the alpha value for the ridge model:

In [ ]:
ridge_alpha_results = []

for alpha in [901, 902 ,903, 904, 905]:
    candidate = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=alpha))])

    result = {"alpha": alpha}

    for year in [2022, 2023, 2024]:
        predictions = walk_forward(candidate, year=year)
        result[year] = predictions["absolute_error"].mean()

    result["Average MAE"] = (result[2022] + result[2023] + result[2024]) / 3
    ridge_alpha_results.append(result)

ridge_alpha_comparison = pd.DataFrame(ridge_alpha_results).sort_values("Average MAE")

display(ridge_alpha_comparison.round(3))

best_alpha = float(ridge_alpha_comparison.iloc[0]["alpha"])     # use the firts line, since its sorted for average MAE and take the alpha value
print(f"Best tested alpha: {best_alpha:g}")

,alpha,2022,2023,2024,Average MAE
2,903,3.548,3.208,2.739,3.165
3,904,3.548,3.208,2.739,3.165
4,905,3.548,3.208,2.739,3.165
0,901,3.548,3.213,2.739,3.167
1,902,3.548,3.213,2.739,3.167


Best tested alpha: 903


#### 2025 Test

In [ ]:
# create a pipeline for the ridge model with its best alpha value
ridge_pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=best_alpha))])


# create a new prediction with our function
ridge_predictions_2025 = walk_forward(ridge_pipeline, year=2025)

# and calculate its MAE
ridge_mae_2025 = ridge_predictions_2025["absolute_error"].mean()

xgboost_seed_maes_2025 = []

# for every seed we fit and predict the results with the hyperparameter from the optuna study
for seed in [0, 13, 123]:
    xgboost_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", XGBRegressor(
            **best_parameters, objective="reg:absoluteerror", 
            tree_method="auto", random_state=seed, n_jobs=2))])

    predictions = walk_forward(xgboost_pipeline, year=2025)
    xgboost_seed_maes_2025.append(predictions["absolute_error"].mean())

results_2025 = [{"model": "Ridge", "mae": ridge_mae_2025}, {"model": "XGBoost (mean across seeds)", 
                "mae": sum(xgboost_seed_maes_2025) / len(xgboost_seed_maes_2025)}]


# creates the data structure for the baseline check. take only the data from the year 2025
evaluation_data = test_data.sort_values(["raceId", "driverId"])

# Starting-grid baseline
grid_scores = evaluation_data["gridPositionNumber"].fillna(float("inf"))            # again append the drivers with missing values at the end
grid_positions = grid_scores.groupby(evaluation_data["raceId"]).rank(method="first")        # rank drivers by their starting position within each race.

results_2025.append({"model": "Starting-grid baseline",
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], grid_positions)})

# Qualifying baseline
quali_scores = evaluation_data["qualificationPositionNumber"].fillna(float("inf"))
quali_positions = quali_scores.groupby(evaluation_data["raceId"]).rank(method="first")

results_2025.append({"model": "Qualifying baseline", 
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], quali_positions)})

comparison_2025 = pd.DataFrame(results_2025).sort_values("mae")
display(comparison_2025.round(3))

,model,mae
1,XGBoost (mean across seeds),3.289
3,Qualifying baseline,3.312
0,Ridge,3.317
2,Starting-grid baseline,3.333


## Current 2026 predictions

In [ ]:
# new test data for the year 2026
test_data_2026 = model_data[model_data["year"] == 2026].copy()

# create a pipeline for the ridge model with its best alpha value
ridge_pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=best_alpha))])


# create a new prediction with our function
ridge_predictions_2026 = walk_forward(ridge_pipeline, year=2026)

# and calculate its MAE
ridge_mae_2026 = ridge_predictions_2026["absolute_error"].mean()

xgboost_seed_maes_2026 = []

# for every seed we fit and predict the results with the hyperparameter from the optuna study
for seed in [0, 13, 123]:
    xgboost_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", XGBRegressor(
            **best_parameters, objective="reg:absoluteerror", 
            tree_method="auto", random_state=seed, n_jobs=2))])

    predictions = walk_forward(xgboost_pipeline, year=2026)
    xgboost_seed_maes_2026.append(predictions["absolute_error"].mean())

results_2026 = [{"model": "Ridge", "mae": ridge_mae_2026}, {"model": "XGBoost (mean across seeds)", 
                "mae": sum(xgboost_seed_maes_2026) / len(xgboost_seed_maes_2026)}]


# creates the data structure for the baseline check. take only the data from the year 2026
evaluation_data = test_data_2026.sort_values(["raceId", "driverId"])

# Starting-grid baseline
grid_scores = evaluation_data["gridPositionNumber"].fillna(float("inf"))            # again append the drivers with missing values at the end
grid_positions = grid_scores.groupby(evaluation_data["raceId"]).rank(method="first")        # rank drivers by their starting position within each race.

results_2026.append({"model": "Starting-grid baseline",
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], grid_positions)})

# Qualifying baseline
quali_scores = evaluation_data["qualificationPositionNumber"].fillna(float("inf"))
quali_positions = quali_scores.groupby(evaluation_data["raceId"]).rank(method="first")

results_2026.append({"model": "Qualifying baseline", 
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], quali_positions)})

comparison_2026 = pd.DataFrame(results_2026).sort_values("mae")
display(comparison_2026.round(3))

,model,mae
3,Qualifying baseline,3.354
1,XGBoost (mean across seeds),3.421
0,Ridge,3.445
2,Starting-grid baseline,3.478


## Add current race weekend data

#### Qualifying Data

In [ ]:
# creates clean float numbers contrary to float
from decimal import Decimal

# let's us intercept errors in download
from urllib.error import HTTPError

# request and open web calls
from urllib.request import Request, urlopen

import pandas as pd

# since the prereleased f1db data is written in YAML-files
import yaml


# create a function to translate a YAML file into a python one
def load_yaml(url):
    request = Request(url)


    with urlopen(request, timeout=60) as response:              # with makes sure that after 60 seconds the request is terminated

        # yaml.safe_load(): creates a python sctucture out of YAML
        # decode(utf-8): translates into text
        return yaml.safe_load(response.read().decode("utf-8"))  


# since the normal file uses milliseconds, we have to translate lap times as 1:33.813
def time_to_milliseconds(value):

    # if a driver hasn't take part in e.g. Q3, his time ramains missing. Not too important, since Q3 and Q2 are currently unused anyway
    if value is None or pd.isna(value):
        return None

    minutes, seconds = str(value).split(":")

    return int((Decimal(minutes) * 60 + Decimal(seconds)) * 1000)


def load_race_data(year, race_folder):
    base_url = ("https://raw.githubusercontent.com/f1db/f1db/main/" + f"src/data/seasons/{year}/races/{race_folder}")

    # uses the first helper function to load the yaml. Here we load prmariliy the raceId and round information
    race = load_yaml(f"{base_url}/race.yml")

    # tries to load the qualifying data 
    try:
        qualifying = load_yaml(f"{base_url}/qualifying-results.yml")
    except HTTPError as error:
        if error.code == 404:   # with this URL no data was found
            raise ValueError("No qualifying file found. Check if the right race folder is selected or try again after F1DB has published the results.")
        raise       # raises the original error, if it isn't 404. If there was no raise here, HTTPError would intercept it but nothing would get displayed

    # check the content of the loaded url
    if not isinstance(qualifying, list) or not qualifying:
        raise ValueError("The qualifying file contains no results.")

    # creates a dataframe form the qualifying data which was until now a lsit of dicts
    drivers = pd.DataFrame(qualifying)

    # add important variables with the same name as in the true race file
    drivers["raceId"] = race["id"]
    drivers["year"] = year
    drivers["round"] = race["round"]

    # creates a numeric position out of the position information. Any values that can't be transformed to an int are transformed to a NaN
    drivers["qualificationPositionNumber"] = pd.to_numeric(drivers["position"], errors="coerce")


    # create different variables for the different qualifying stages
    for stage in [1, 2, 3]:
        column = f"q{stage}"

        # if a whole qualifying stage would be missing this prevents a KeyError
        if column not in drivers.columns:
            drivers[column] = None

        # create the timed columns with their correct name. 
        # map: calls the second helper function for each value
        # Int64: is pandas integer type which allows for missing values to occur
        drivers[f"{column}Millis"] = drivers[column].map(time_to_milliseconds).astype("Int64")

    # select the used columns
    columns = [
        "raceId",
        "year",
        "round",
        "driverId",
        "constructorId",
        "qualificationPositionNumber",
        "q1Millis",
        "q2Millis",
        "q3Millis",
    ]

    drivers = drivers[columns]

    # test for duplicated drivers
    if drivers.duplicated(["raceId", "driverId"]).any():
        raise ValueError("Duplicate qualifying entries")

    return race, drivers


#### Starting Grid
Currently has to be typed in manually

In [ ]:
from pathlib import Path
from data import race_data

YEAR = 2026
RACE_FOLDER = "19-mexico"  # Change this to the desired race folder.

# assign the right race_info and participants variables
# race_info: all the information from the load_race_data
# participans: the drivers variable from load_race_data, i.e. the additional qualifiying inforamtion
race_info, participants = race_data.load_race_data(year=YEAR, race_folder=RACE_FOLDER)

# crates the grid for the new csv file to type in the starting position manually
grid_path = (
    PROJECT_DIR / "data" / "input"
    / str(YEAR) / RACE_FOLDER / "starting_grid.csv"
)

# create the neede parent folders. if they already exist the will be left as they are
grid_path.parent.mkdir(parents=True, exist_ok=True)

# create a template for the values that have to be added manually
# if not ....exists() prevents that the file has to be filled out again if  function is called again
if not grid_path.exists():
    starting_grid = participants[["raceId", "driverId"]].copy()
    starting_grid["gridPositionText"] = ""

    # stores the starting_grid file as a csv in the given path, without pandas creating new indizes
    starting_grid.to_csv(grid_path, index=False)

In [ ]:
# we ned a function to add the starting grid manually. Our qualifying data return: drivers will be used as the participants parameter input
# grid path is the path to the new grid csv
def add_starting_grid(participants, grid_path):

    # read_csv: as we created the csv with the starting position contained, we now have to read it in again
    # dtype: starting pos is read as a string. thus the same column can have values like "1" or "PL"
    grid = pd.read_csv(grid_path, dtype={"gridPositionText": "string"})

    # clean it up
    grid["gridPositionText"] = grid["gridPositionText"].str.strip().str.upper()

    # require exactly one grid entry for every qualifying participant
    drivers = participants.merge(grid[["raceId", "driverId", "gridPositionText"]],
        on=["raceId", "driverId"],
        how="outer",
        validate="one_to_one",
        indicator=True)         # creates a helper column: _merge. She shows where the column comes from -> both, left_only or right_only

    # test, if all the drivers form the qualifiyng start or if a new one starts
    if not drivers["_merge"].eq("both").all():
        raise ValueError("The starting-grid file does not match the participants.")

    # get rid of the helper column
    drivers = drivers.drop(columns="_merge")

    # accept positive grid numbers or explicitly recorded special cases
    text = drivers["gridPositionText"]

    # again use "coerce" to allow for non int values
    numbers = pd.to_numeric(text, errors="coerce")

    # numbers must be greater or equal to 1 and have to be divisible through 1 without a remainder, i.e. they are positive integers
    valid_number = numbers.ge(1) & numbers.mod(1).eq(0)
    valid_special = text.isin(["PL", "DNS", "DNQ"])

    # the function stops, if a value contained an invalid number/value
    if not (valid_number | valid_special).fillna(False).all():
        raise ValueError("Fill every starting position with a positive integer PL, DNS or DNQ.")

    # exclude only explicitly confirmed non-starters, i.e. drivers with a DNS or DNQ in their starting value
    # ~ swaps all the selection
    # .copy(), because we want to add gridPositionText, without causing any sideeffect in the participans parameter
    drivers = drivers.loc[~text.isin(["DNS", "DNQ"])].copy()


    # compute the field size
    drivers["fieldSize"] = drivers.groupby("raceId")["driverId"].transform("size")

    # .loc[drivers.index] chooses only the drivers that remain after excluding DNS & DNQ
    drivers["gridPositionNumber"] = numbers.loc[drivers.index]

    # Use the same pitlane representation as above in the notebook
    drivers.loc[drivers["gridPositionText"].eq("PL"), "gridPositionNumber"] = drivers["fieldSize"]

    return drivers

### How is the performance without the starting grid inforamtion

In [138]:
import pandas as pd
import importlib
from models import features
from models.model import create_model

importlib.reload(features)

data = features.prepare_features()
data["date"] = pd.to_datetime(data["date"])

columns = features.numeric_features + features.categorical_features
results = []

# evaluate each race separately, including all recorded participants
evaluation_data = data.loc[data["year"].between(2022, 2024)]

for race_id, race in evaluation_data.groupby("raceId"):
    race = race.sort_values("driverId").copy()
    race_date = race["date"].iloc[0]

    # train only on earlier races within the previous ten years
    train = data.loc[
        data["date"].ge(race_date - pd.DateOffset(years=10))
        & data["date"].lt(race_date)
        & data["finish_fraction"].notna()
    ]

    model = create_model(
        features.numeric_features,
        features.categorical_features,
    )
    model.fit(train[columns], train["finish_fraction"])

    # rank everyone before excluding non-starters from scoring, as we don't know before the race has started, whether they start or not
    race["ridge_score"] = model.predict(race[columns])
    race["ridge_position"] = race["ridge_score"].rank(method="first")

    race["baseline_position"] = (
        race["qualificationPositionNumber"]
        .fillna(float("inf"))
        .rank(method="first")
    )

    # non-starters have no valid finishing-position target, 
    # thus they are omited as they give no interesting informaiton on our model performance
    scored = race.loc[race["finish_fraction"].notna()]

    results.append({
        "raceId": race_id,
        "Ridge without grid": (
            scored["ridge_position"] - scored["positionDisplayOrder"]
        ).abs().mean(),
        "Qualifying baseline": (
            scored["baseline_position"] - scored["positionDisplayOrder"]
        ).abs().mean(),
    })

race_errors = pd.DataFrame(results).set_index("raceId")

print("Races evaluated:", len(race_errors))
display(race_errors.mean().rename("MAE").to_frame().round(3))

Races evaluated: 68


,MAE
Ridge without grid,3.175
Qualifying baseline,3.354


# Weekly Model

Ridge:

In [142]:
import pandas as pd
import importlib
from models import features
from models.model import create_model

importlib.reload(features)

early_data = features.prepare_features()
early_data["date"] = pd.to_datetime(early_data["date"])

numeric_columns = features.pre_race_numeric_features
columns = numeric_columns + features.categorical_features

early_results = []
score_error_rows = []

for race_id, race in early_data.loc[
    early_data["year"].between(2022, 2024)
].groupby("raceId"):
    race = race.sort_values("driverId").copy()
    race_date = race["date"].iloc[0]

    # train only on earlier races within the previous ten years
    train = early_data.loc[
        early_data["date"].ge(race_date - pd.DateOffset(years=10))
        & early_data["date"].lt(race_date)
        & early_data["finish_fraction"].notna()
    ]

    model = create_model(
        numeric_columns,
        features.categorical_features,
    )
    model.fit(train[columns], train["finish_fraction"])

    # predict without any current qualifying features
    race["ridge_score"] = model.predict(race[columns])
    race["ridge_position"] = race["ridge_score"].rank(method="first")

    # use recent driver form, with team form as a fallback
    baseline_scores = (
        race["driver_finish_5"]
        .fillna(race["team_finish_5"])
        .fillna(train["finish_fraction"].median())
    )
    race["baseline_position"] = baseline_scores.rank(method="first")

    # score only participants with a valid finishing-position target
    scored = race.loc[race["finish_fraction"].notna()]


    # collect errors from races that were not used to train the model
    race_errors = scored[[
        "raceId",
        "year",
        "date",
        "driverId",
        "constructorId",
        "finish_fraction",
        "ridge_score",
    ]].copy()

    race_errors["score_error"] = race_errors["finish_fraction"] - race_errors["ridge_score"]
    
    score_error_rows.append(race_errors)

    early_results.append({
        "raceId": race_id,
        "Pre-race Ridge": (
            scored["ridge_position"] - scored["positionDisplayOrder"]
        ).abs().mean(),
        "Recent-form baseline": (
            scored["baseline_position"] - scored["positionDisplayOrder"]
        ).abs().mean(),
    })

early_errors = pd.DataFrame(early_results).set_index("raceId")

print("Races evaluated:", len(early_errors))
display(early_errors.mean().rename("MAE").to_frame().round(3))


# combine the errors from all validation races
validation_errors = pd.concat(score_error_rows, ignore_index=True)

# remove shared race-level shifts that do not affect the ranking
# thus we subtract the mean to eliminate the noise of the race, to only use the individual noise.
validation_errors["ranking_error"] = (
    validation_errors["score_error"]
    - validation_errors.groupby("raceId")["score_error"].transform("mean")
)

# estimate an initial noise scale for the season simulation
score_noise_std = validation_errors["ranking_error"].std()

# save the validation errors for later simulation and checks
evaluation_dir = PROJECT_DIR / "data" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

validation_errors.to_csv(
    evaluation_dir / "pre_race_errors_2022_2024.csv",
    index=False,
)

print("Validation races:", validation_errors["raceId"].nunique())
print("Score noise standard deviation:", round(score_noise_std, 4))

Races evaluated: 68


,MAE
Pre-race Ridge,3.624
Recent-form baseline,3.916


Validation races: 68
Score noise standard deviation: 0.2328


We estimate uncertainty using historical walk-forward prediction errors on the model’s score scale. We subtract each race’s mean error so that shared shifts, which do not affect driver rankings, do not inflate the estimated uncertainty. We then calculate the standard deviation of the remaining errors.

For each simulation, we add normally distributed noise with this standard deviation to the predicted scores, rank the drivers and award points. Repeating this for the remaining races and sprints produces estimated driver and constructor championship probabilities.

XGBoost:

In [140]:
from xgboost import XGBRegressor

xgb_columns = (
    features.pre_race_numeric_features
    + features.categorical_features
)

xgb_results = []

for race_id, race in early_data.loc[
    early_data["year"].between(2022, 2024)
].groupby("raceId"):
    race = race.sort_values("driverId").copy()
    race_date = race["date"].iloc[0]

    # use the same training window as the ridge evaluation
    train = early_data.loc[
        early_data["date"].ge(race_date - pd.DateOffset(years=10))
        & early_data["date"].lt(race_date)
        & early_data["finish_fraction"].notna()
    ]

    # keep the existing preprocessing and replace only the regressor
    model = create_model(
        features.pre_race_numeric_features,
        features.categorical_features,
    )
    model.set_params(
        regression=XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=10,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=2,
        )
    )

    model.fit(train[xgb_columns], train["finish_fraction"])

    # rank all participants before selecting valid evaluation targets
    race["xgb_score"] = model.predict(race[xgb_columns])
    race["xgb_position"] = race["xgb_score"].rank(method="first")

    scored = race.loc[race["finish_fraction"].notna()]

    xgb_results.append({
        "raceId": race_id,
        "Pre-race XGBoost": (
            scored["xgb_position"] - scored["positionDisplayOrder"]
        ).abs().mean(),
    })

xgb_errors = pd.DataFrame(xgb_results).set_index("raceId")

comparison = early_errors.join(xgb_errors, how="inner")

print("Races evaluated:", len(comparison))
display(comparison.mean().rename("MAE").to_frame().round(3))

Races evaluated: 68


,MAE
Pre-race Ridge,3.624
Recent-form baseline,3.916
Pre-race XGBoost,3.678


Thus it seems as if the ridge model also works on weekly predictions